# freeze-requires-grad — ex2: selectively unfreeze the last N submodules of an encoder

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `freeze-requires-grad`. Running the final beacon cell reports progress against the `PyTorch: freeze via requires_grad=False` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: freeze via requires_grad=False` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`freeze-requires-grad`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "freeze-requires-grad"
DD_SUBTOPIC = "PyTorch: freeze via requires_grad=False"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Selective unfreeze of the last N submodules — quick refresher

Pure 'freeze backbone, train head' is the simplest transfer-learning pattern. A finer-grained alternative: freeze EVERYTHING then **re-enable** training on only the last `n_last` submodules of the encoder. This trains a bit more of the network than head-only, but much less than full fine-tune.

```
for p in model.parameters():
    p.requires_grad = False                              # freeze all
children = list(model.encoder.children())                # ordered list
for layer in children[-n_last:]:                         # last n_last submodules
    for p in layer.parameters():
        p.requires_grad = True                           # unfreeze
```

**Exemplar.** For `encoder = Sequential(L0, L1, L2, L3)` and `n_last = 2`, layers `L2` and `L3` regain `requires_grad=True`; `L0` and `L1` stay frozen.

### Exercise 2 — selectively unfreeze the last N submodules of an encoder

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the partial fine-tune pattern: freeze every param, then re-enable `requires_grad=True` on the parameters of the last `n_last` children of an encoder Sequential.
> Keywords: transfer-learning, partial-fine-tune, requires_grad, encoder-children
> ```

**KCs targeted:** `freeze-requires-grad-false`, `partial-unfreeze-last-n`

A toy 'pretrained' model is provided:

```
class ToyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(10, 32),     # children[0]
            nn.ReLU(),             # children[1] — no params
            nn.Linear(32, 16),     # children[2]
            nn.ReLU(),             # children[3] — no params
            nn.Linear(16, 16),     # children[4]
        )
        self.fc = nn.Linear(16, 5)
    def forward(self, x):
        return self.fc(self.encoder(x))
```

Implement `ex2_partial_unfreeze(model, n_last)` that performs **partial fine-tuning** in this order:

1. **Freeze everything**: set `p.requires_grad = False` on every param of `model`.
2. **Re-enable the last `n_last` encoder children**: take `list(model.encoder.children())[-n_last:]` and for each submodule set `p.requires_grad = True` on every param inside.
3. Leave the head `model.fc` FROZEN — that's the difference from the ex1 'replace head' pattern.
4. Return the list of trainable params (`[p for p in model.parameters() if p.requires_grad]`).

**Indexing detail.** `list(model.encoder.children())[-n_last:]` grabs the LAST `n_last` submodules regardless of which are param-bearing. ReLU children have no params, so they contribute nothing to the trainable list — but they still count toward the slice. This matches the standard 'last 2 blocks' partial-tune pattern.

**Boundary.** Assume `1 <= n_last <= len(list(model.encoder.children()))`.

In [ ]:
def ex2_partial_unfreeze(model, n_last: int):
    for p in model.parameters():
        p.requires_grad = False
    children = list(model.encoder.children())
    for layer in children[-n_last:]:
        for p in layer.parameters():
            p.requires_grad = True
    return [p for p in model.parameters() if p.requires_grad]


<details><summary>Solution</summary>

```python
def ex2_partial_unfreeze(model, n_last: int):
    for p in model.parameters():
        p.requires_grad = False
    children = list(model.encoder.children())
    for layer in children[-n_last:]:
        for p in layer.parameters():
            p.requires_grad = True
    return [p for p in model.parameters() if p.requires_grad]
```

**Why `list(model.encoder.children())[-n_last:]` and not `model.encoder[-n_last:]`.** Both work for `nn.Sequential`, but `.children()` is the generic interface across any `nn.Module` container — it'd work for a custom encoder built from named submodules too. The slice is the portable form.

**Why ReLU in the slice is harmless.** `nn.ReLU.parameters()` yields nothing, so `for p in relu.parameters(): p.requires_grad = True` is a no-op. The slice can include any mix of param-bearing and parameter-less modules without affecting correctness.

**Contrast with ex1.** Ex1 also replaces `model.fc` with a fresh head — new modules default to `requires_grad=True`, so the head trains. Here we LEAVE the original head in place and DON'T unfreeze it. Use case: continue using the original head (e.g. dimensionality is fine) but fine-tune deeper into the network. Common when transferring within-domain (same label space, different distribution).

**Generalization to ResNet.** Replace `model.encoder` with `model.layer4` (the last residual stage) and you have the standard 'fine-tune the last block' recipe used in countless downstream-task papers.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()